In [ ]:
!pip install torch_geometric
!pip install wandb

In [ ]:
import os
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr
from torch.utils.data import Dataset
from torch_geometric.loader import DataLoader
from tqdm import tqdm

import torch
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv, global_mean_pool
from torch.amp import autocast, GradScaler

import wandb
import gc

In [ ]:
path_main = ''
os.chdir(path_main)

In [ ]:
# model Class
class ProBAN_Net(torch.nn.Module):
    def __init__(self, node_dim=1282, edge_dim=4, global_dim=3, hidden_dim=128):
        super(ProBAN_Net, self).__init__()

        self.node_encoder = torch.nn.Linear(node_dim, hidden_dim)
        self.use_structure = True #


        self.conv1 = GATv2Conv(hidden_dim, hidden_dim, heads=1, edge_dim=edge_dim)
        self.conv2 = GATv2Conv(hidden_dim, hidden_dim, heads=1, edge_dim=edge_dim)

        # RMSF prediction head
        self.rmsf_head = torch.nn.Linear(hidden_dim, 1)

        # The main head
        self.dG_head = torch.nn.Sequential(
            torch.nn.Linear(hidden_dim + global_dim, 64),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.2),
            torch.nn.Linear(64, 1)
        )

    def forward(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        global_features = data.global_features

        # If the flag is off, replace the edges with zeros.
        if not self.use_structure:
            edge_attr = torch.zeros_like(edge_attr)

        x = F.relu(self.node_encoder(x))
        x = F.elu(self.conv1(x, edge_index, edge_attr))
        x = F.elu(self.conv2(x, edge_index, edge_attr))

        pred_rmsf = self.rmsf_head(x)

        global_repr = global_mean_pool(x, batch)
        global_repr = torch.cat([global_repr, global_features], dim=-1)
        pred_dG = self.dG_head(global_repr)

        return pred_dG, pred_rmsf



In [ ]:
# Dataset Class
# BENCHMARK CLASS (With edge filtering and correct target)
class ProBANBenchmarkDataset_cut(Dataset):
    def __init__(self, csv_file, data_dir, cutoff=15.0, distance_col_idx=0):
        """
        :param csv_file: Path to the Excel/CSV table with true values
        :param data_dir: Folder containing .pt graph files
        :param cutoff: Maximum distance to keep an edge (in Angstroms)
        :param distance_col_idx: Index of the distance column in edge_attr
        """
        print(f"📊 Reading benchmark table: {csv_file}")
        self.data_dir = data_dir
        self.cutoff = cutoff
        self.distance_col_idx = distance_col_idx

        raw_df = pd.read_excel(csv_file).dropna(subset=['pdb_id', 'dG'])

        # Keep only those complexes for which graphs physically exist
        valid_rows = []
        for _, row in raw_df.iterrows():
            pdb_id = str(row['pdb_id']).strip().lower()
            if os.path.exists(os.path.join(self.data_dir, f"{pdb_id}.pt")):
                valid_rows.append(row)

        self.df = pd.DataFrame(valid_rows)
        print(f"✅ Ready graphs for testing: {len(self.df)} out of {len(raw_df)}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        pdb_id = str(row['pdb_id']).strip().lower()

        # Get the TRUE dG value from the table
        # (If you previously divided by -1.3643 to convert to pKd,
        # uncomment this action if it is needed for your model)
        dg_value = float(row['dG'])
        # dg_value = float(row['dG']) / -1.3643

        file_path = os.path.join(self.data_dir, f"{pdb_id}.pt")
        data = torch.load(file_path, map_location='cpu', weights_only=False)

        # --- PATCH: Edge filtering by distance (Cutoff) ---
        if data.edge_attr is not None and data.edge_index is not None:
            distances = data.edge_attr[:, self.distance_col_idx]
            mask = distances <= self.cutoff

            data.edge_index = data.edge_index[:, mask]
            data.edge_attr = data.edge_attr[mask]

        # Save the true dG in the graph
        data.true_dG = torch.tensor([dg_value], dtype=torch.float32)

        # --- IMPORTANT FIX: Pass the true target for the Loss function ---
        # Now the model will evaluate its error relative to the real dG, not 0.0
        data.y_mut_delta_delta = torch.tensor([dg_value], dtype=torch.float32)

        # Dummy variables for auxiliary tasks
        data.y_rmsf = torch.zeros((data.x.shape[0], 1))
        data.has_md = torch.tensor([False], dtype=torch.bool)
        data.item_id = pdb_id

        return data

In [ ]:

# TESTING AND CONVERSION FUNCTION
def run_benchmark_inference(model_class, weights_path, csv_path, data_dir, device):
    print("\n Preparing dataloader...")
    test_dataset = ProBANBenchmarkDataset_cut(csv_path, data_dir)
    test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

    print("\n Initializing model...")
    model = model_class().to(device)
    #model.use_structure = False

    if not os.path.exists(weights_path):
        raise FileNotFoundError(f"Weights '{weights_path}' not found!")

    model.load_state_dict(torch.load(weights_path, map_location=device))
    model.eval()

    true_dG_list = []
    pred_dG_list = []

    # Thermodynamic constant at 298.15 K
    RT_LN10 = -1.3643

    print("Running graphs through the network...")
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Inference"):
            batch = batch.to(device)

            # 1. The network makes a prediction in its native scale (pKd)
            pred_pKd, _ = model(batch)

            # 2. PHYSICAL CONVERSION: pKd -> dG (kcal/mol)
            pred_dG = pred_pKd * RT_LN10

            # 3. Save the results
            true_dG_list.extend(batch.true_dG.cpu().numpy())
            pred_dG_list.extend(pred_dG.view(-1).cpu().numpy())

    true_dG = np.array(true_dG_list)
    pred_dG = np.array(pred_dG_list)

    # METRIC CALCULATION
    rmse = np.sqrt(np.mean((true_dG - pred_dG)**2))
    mae = np.mean(np.abs(true_dG - pred_dG))
    pearson_r, _ = pearsonr(true_dG, pred_dG)
    spearman_r, _ = spearmanr(true_dG, pred_dG)

    print("\n" + "-"*45)
    print("Test_A_B BENCHMARK METRICS (in kcal/mol)")
    print("="*45)
    print(f"Pearson R:  {pearson_r:.4f}")
    print(f"Spearman R: {spearman_r:.4f}")
    print(f"MAE:        {mae:.4f} kcal/mol")
    print(f"RMSE:       {rmse:.4f} kcal/mol")

    # VISUALIZATION
    plt.figure(figsize=(8, 8))
    sns.set_theme(style="whitegrid", context="paper", font_scale=1.3)

    plt.scatter(true_dG, pred_dG, alpha=0.75, color='#1f77b4', edgecolors='w', s=80, linewidth=1.5)

    # Ideal diagonal
    min_val = min(true_dG.min(), pred_dG.min()) - 1.0
    max_val = max(true_dG.max(), pred_dG.max()) + 1.0
    plt.plot([min_val, max_val], [min_val, max_val], 'k--', lw=2, label='Ideal match')



    plt.title(f'ProBAN 2.0 Test_A_B Benchmark\n$R_p$ = {pearson_r:.3f} | MAE = {mae:.2f} kcal/mol', pad=20, fontweight='bold')
    plt.xlabel('Experimental $\Delta G$ [kcal/mol]', fontweight='bold')
    plt.ylabel('Predicted $\Delta G$ [kcal/mol]', fontweight='bold')


    plt.xlim(max_val, min_val)
    plt.ylim(max_val, min_val)

    plt.legend(loc='lower right', frameon=True, shadow=True)
    plt.tight_layout()
    plt.savefig(f"Test_A_B/Test_A_B_Results_dG{MODEL_WEIGHTS}.png", dpi=300)
    plt.show()



In [ ]:
TEST_CSV_PATH = "Test_A_B/Test_A_B.xlsx"
TEST_GRAPHS_DIR = "./Test_A_B/benchmark_graphs"
MODEL_WEIGHTS = "best_proban_model_26_bc4_cut_15_mix.pth"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# EXECUTION

if __name__ == "__main__":
    run_benchmark_inference(
        model_class=ProBAN_Net,
        weights_path=MODEL_WEIGHTS,
        csv_path=TEST_CSV_PATH,
        data_dir=TEST_GRAPHS_DIR,
        device=device
    )